#### Imports

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix
from sklearn.model_selection import train_test_split
from torch.utils.data import Subset
from datasets import load_dataset
import re
import torch
from transformers import pipeline, AutoTokenizer
import matplotlib.pyplot as plt
import numpy as np
import os
import json
from datetime import datetime

#### Data Retrieval

In [ ]:
seed = 42

# Load UltraFeedback Dataset
ds = load_dataset("openbmb/UltraFeedback")

print(ds)

README.md: 0.00B [00:00, ?B/s]

evol_instruct.jsonl:   0%|          | 0.00/168M [00:00<?, ?B/s]

false_qa.jsonl:   0%|          | 0.00/25.9M [00:00<?, ?B/s]

flan.jsonl:   0%|          | 0.00/240M [00:00<?, ?B/s]

sharegpt.jsonl:   0%|          | 0.00/313M [00:00<?, ?B/s]

truthful_qa.jsonl: 0.00B [00:00, ?B/s]

ultrachat.jsonl:   0%|          | 0.00/182M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/63967 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['source', 'instruction', 'models', 'completions', 'correct_answers', 'incorrect_answers'],
        num_rows: 63967
    })
})


#### Load Principles

In [ ]:
# Open Valid Principles Extracted from Experiment Runs
with open("experiment_runs/principles_valid.json", "r") as f:
    data = json.load(f)

# Transform them into a list of principles with their name, description, and query
principles_all = []
for run in data:
    for principle in run["specific_criteria"]:
        principles_all.append({
            "name": principle["name"],
            "description": principle["definition"],
            "query": run["query"]
        })

# Save the principles to a new JSON file
with open("experiment_runs/raw_principles.json", "w", encoding="utf-8") as f:
    json.dump(principles_all, f, ensure_ascii=False, indent=2)

print(f"{len(principles_all)} principles extracted and saved to experiment_runs/raw_principles.json")

#### Deepseek Transformer Pipe Initial Setup

In [ ]:
pipe = pipeline(
    "text-generation",
    model="deepseek-ai/DeepSeek-R1-Distill-Qwen-7B",
    device_map="auto"
)

messages = [
    {"role": "user", "content": "Who are you?"}
]

result = pipe(messages, max_new_tokens=200)

with open("experiment_runs/initial_deepseek_test.json", "w", encoding="utf-8") as f:
            json.dump(result, f, ensure_ascii=False, indent=2)

print(result[0]["generated_text"])

#### Deepseek Interpretter Test

In [ ]:
N = 200
ds_subset = ds["train"].shuffle(seed=108).select(range(N))
np.random.seed(108)
principle_subset = np.random.permutation(np.array(principles_all))[:N]

principle_interpretted_runs = []

for i in range(N):
    query = ds_subset[i]["instruction"]
    responses = ds_subset[i]['correct_answers'] + ds_subset[i]['incorrect_answers']
    if len(responses) == 0:
        continue
    response = responses[0]
    principle = principle_subset[i]

    prompt = f"""
    Role: You are an evaluator.

    Task:
    Score how well the RESPONSE applies the PRINCIPLE while answering the QUESTION.

    Evaluation Criteria:
    - Score from 0 to 10.
    - 0 = completely violates, ignores, or misapplies the principle.
    - 5 = partially applies the principle, but with significant weaknesses, overuse, or inappropriate usage.
    - 10 = applies the principle appropriately, consistently, and in a context-sensitive manner.

    Instructions:
    - Evaluate ONLY the application of the PRINCIPLE.
    - The "Reference Query" describes the original context in which the PRINCIPLE was learned or derived.
    - The "Reference Query" defines the intended scope, conditions, and usage pattern of the PRINCIPLE.
    - Principles are NOT universally applicable. Some principles are only appropriate in certain contexts.
    - Evaluate whether the RESPONSE applies the PRINCIPLE appropriately for the actual QUESTION being answered.
    - A RESPONSE should receive a lower score if it:
    - applies the principle in situations where it is not appropriate,
    - overgeneralizes the principle beyond its intended context,
    - forces the principle unnaturally,
    - or ignores contextual differences between the QUESTION and the Reference Query.
    - Use the Reference Query to infer:
    - when the principle should be applied,
    - how strongly it should be applied,
    - and what type of behavior the principle is intended to encourage.
    - Do NOT treat the "Reference Query" as the QUESTION being answered.
    - Do NOT evaluate factual correctness unless it directly affects the principle.
    - Do NOT explain your reasoning.
    - Do NOT output anything except a single integer from 0 to 10.

    QUESTION:
    {query}

    PRINCIPLE:
    Name:
    {principle["name"]}

    Description:
    {principle["description"]}

    Reference Query:
    {principle["query"]}

    RESPONSE:
    {response}
    """
    
    messages = [
        {"role": "user", "content": prompt}
    ]
    
    result = pipe(messages, max_new_tokens=1000)
    
    score = result[0]["generated_text"]
    
    principle_interpretted_runs.append({
        "query": query,
        "response": response,
        "principle_name": principle["name"],
        "principle_description": principle["description"],
        "principle_query": principle["query"],
        "score": score
    })

    if i == 0:
        with open("experiment_runs/initial_run.json", "w", encoding="utf-8") as f:
            json.dump(principle_interpretted_runs, f, ensure_ascii=False, indent=2)

    print(f"Processed {i+1}/{N} principles.")

with open("experiment_runs/interpreted_runs.json", "w", encoding="utf-8") as f:
    json.dump(principle_interpretted_runs, f, ensure_ascii=False, indent=2)